# 05 — API Server: OpenAI-Compatible Endpoint via Cloudflare Tunnel

This is the **main API server notebook**. It:

1. Installs `llama-cpp-python` with CUDA
2. Downloads a model (default: **Qwen3.6-35B-A3B APEX Nano**)
3. Starts a **FastAPI server** wrapping `llama_cpp.Llama`, exposing `/v1/chat/completions`
4. Downloads **cloudflared** and creates a **public tunnel**
5. Prints the public URL — use it as your OpenAI `base_url` from anywhere

The result: a free, public, OpenAI-compatible LLM API running on a Colab T4.

## 1. Install dependencies

In [ ]:
%%capture
pip install llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
pip install fastapi uvicorn[standard] pydantic


## 2. Download the model

In [ ]:
from huggingface_hub import hf_hub_download
import os

# Default: Qwen3.6-35B-A3B APEX Nano (35B MoE, fits in 15GB VRAM)
# Change repo_id / filename to use a different model
repo_id = "mudler/Carnice-Qwen3.6-MoE-35B-A3B-APEX-MTP-GGUF"

from huggingface_hub import list_repo_files
files = list_repo_files(repo_id)
gguf_files = [f for f in files if f.endswith('.gguf')]
filename = gguf_files[0] if gguf_files else None
assert filename, f"No GGUF found in {repo_id}"

print(f"Downloading {repo_id} / {filename} ...")
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"Done. Path: {model_path}")
print(f"Size: {os.path.getsize(model_path) / 1024**3:.2f} GB")


## 3. Load the model onto the GPU

In [ ]:
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=8192,
    verbose=False,
)
print(f"Model loaded in {time.time()-t0:.1f}s")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 4. Write the FastAPI server (OpenAI-compatible)

In [ ]:
# This cell writes server.py — a FastAPI app that wraps the loaded Llama model
# and exposes an OpenAI-compatible /v1/chat/completions endpoint.

server_code = '''
import time, uuid, json
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from llama_cpp import Llama
import os

app = FastAPI(title="Colab LLM API")

MODEL_PATH = os.environ.get("MODEL_PATH", "")
llm = None

@app.on_event("startup")
def load_model():
    global llm
    if MODEL_PATH:
        print(f"Loading model from {MODEL_PATH}...")
        llm = Llama(model_path=MODEL_PATH, n_gpu_layers=-1, n_ctx=8192, verbose=False)
        print("Model loaded ✅")
    else:
        print("WARNING: MODEL_PATH not set")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = "default"
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 256
    stream: Optional[bool] = False
    top_p: Optional[float] = 0.95

def build_prompt(messages):
    prompt = ""
    for m in messages:
        if m.role == "system":
            prompt += f"<|im_start|>system\\n{m.content}<|im_end|>\\n"
        elif m.role == "user":
            prompt += f"<|im_start|>user\\n{m.content}<|im_end|>\\n"
        elif m.role == "assistant":
            prompt += f"<|im_start|>assistant\\n{m.content}<|im_end|>\\n"
    prompt += "<|im_start|>assistant\\n"
    return prompt

@app.get("/")
def root():
    return {"status": "ok", "model": "loaded" if llm else "not loaded"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": "colab-model", "object": "model"}]}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    if llm is None:
        raise HTTPException(503, "Model not loaded")
    prompt = build_prompt(req.messages)
    resp = llm(
        prompt,
        max_tokens=req.max_tokens,
        temperature=req.temperature,
        top_p=req.top_p,
        stop=["<|im_end|>", "<|endoftext|>"],
        stream=req.stream,
        verbose=False,
    )
    if req.stream:
        def gen():
            for chunk in resp:
                text = chunk["choices"][0].get("text", "")
                if text:
                    yield f"data: {json.dumps({'choices': [{'delta': {'content': text}}]})}\\n\\n"
            yield "data: [DONE]\\n\\n"
        return StreamingResponse(gen(), media_type="text/event-stream")
    text = resp["choices"][0]["text"]
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }
'''

with open("server.py", "w") as f:
    f.write(server_code)
print("server.py written ✅")


## 5. Start the API server in the background

We export `MODEL_PATH` so the server reuses the already-downloaded model, then launch `uvicorn` with `nohup` so it keeps running in the background.

In [ ]:
import subprocess, os, signal, time

# Kill any existing server
os.system("pkill -f uvicorn || true")

env = os.environ.copy()
env["MODEL_PATH"] = model_path

# Start uvicorn in background
proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8080"],
    env=env,
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(3)  # let it boot
print(f"Server PID: {proc.pid}")

# Quick local test
import requests
r = requests.get("http://localhost:8080/")
print(f"Local health check: {r.status_code} → {r.json()}")


In [ ]:
# Test a chat completion locally
import requests
r = requests.post("http://localhost:8080/v1/chat/completions", json={
    "model": "qwen3.6-35b",
    "messages": [{"role": "user", "content": "Say hello in 5 languages."}],
    "max_tokens": 100,
})
print(r.json()["choices"][0]["message"]["content"])


## 6. Install cloudflared

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


## 7. Start the Cloudflare tunnel

This creates a **public URL** that forwards to `localhost:8080`. The URL is printed in the output — copy it and use it as your OpenAI `base_url`.

> The URL is ephemeral and changes each time you restart the tunnel. It's public — anyone with the link can call your API.

In [ ]:
import subprocess, re, time, threading

# Start cloudflared tunnel in background, capture output
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

tunnel_url = None
start = time.time()
while time.time() - start < 30:
    line = tunnel_proc.stdout.readline().decode(errors="ignore")
    if not line:
        break
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break

if tunnel_url:
    print(f"\n{"="*60}")
    print(f"✅ PUBLIC API URL: {tunnel_url}/v1/chat/completions")
    print(f"✅ Base URL:       {tunnel_url}/v1")
    print(f"{"="*60}")
else:
    print("Tunnel URL not found — check cloudflared output above")


## 8. Test the public endpoint

In [ ]:
if tunnel_url:
    import requests
    r = requests.post(f"{tunnel_url}/v1/chat/completions", json={
        "model": "qwen3.6-35b",
        "messages": [{"role": "user", "content": "What is 17 * 23? Just the number."}],
        "max_tokens": 50,
    })
    print(f"Status: {r.status_code}")
    print(r.json()["choices"][0]["message"]["content"])
else:
    print("No tunnel URL — re-run cell 7")


## 🎉 You're live!

Use the tunnel URL from cell 7 as your OpenAI `base_url`:

```python
from openai import OpenAI
client = OpenAI(base_url="https://<your-url>.trycloudflare.com/v1", api_key="sk-no-key-needed")
```

The tunnel stays alive as long as this Colab notebook is running. To stop: interrupt cell 7 or let the runtime disconnect.

### To switch models
Edit cell 2 to change `repo_id` / `filename`, re-run cells 2–5. The server will reload the new model on startup.